# AirBnB Effect Study — ARIMAX Model
## Step 1: Caricamento e preparazione dati

**Città analizzate:** Roma · Milano · Firenze (Italia) &nbsp;|&nbsp; New York (USA)

| Città | Fonte 2024 | Fonte 2025 | Snapshot totali |
|-------|-----------|-----------|----------------|
| Roma | airbnb.csv (4 trim.) | Inside Airbnb (4 snap.) | 8 |
| Milano | airbnb.csv (4 trim.) | Inside Airbnb (2 snap.) | 6 |
| Firenze | airbnb.csv (4 trim.) | Inside Airbnb (3 snap.) | 7 |
| New York | Kaggle zip (2 snap.) | Inside Airbnb (9 snap.) | 11 |

**Output:** `data/processed/<city>.csv` &nbsp;·&nbsp; `data/processed/master_airbnb.csv`


In [3]:
import pandas as pd
import zipfile
import glob
import os

# Cartella di output
OUTPUT_DIR = 'data/processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Città italiane incluse (Roma e Milano hanno anche dati 2025;
# Firenze è la 3ª città IT per volume con ~41k annunci)
CITIES_IT = ['Roma', 'Milano', 'Firenze']

# Schema colonne comune per tutti i CSV finali
FINAL_COLS = [
    'id', 'city', 'neighbourhood', 'room_type', 'price', 'date',
    'number_of_reviews', 'number_of_reviews_ltm',
    'reviews_per_month', 'availability_365', 'minimum_nights', 'source'
]

print("Output dir:", os.path.abspath(OUTPUT_DIR))
print("Colonne finali:", FINAL_COLS)


Output dir: c:\Users\user\airbnb\AirBnB-effect-study-with-ARIMAX-model\data\processed
Colonne finali: ['id', 'city', 'neighbourhood', 'room_type', 'price', 'date', 'number_of_reviews', 'number_of_reviews_ltm', 'reviews_per_month', 'availability_365', 'minimum_nights', 'source']


## Funzioni di caricamento

Tre funzioni, una per fonte:
- **`load_italy_2024`** — legge `airbnb.csv` (formato custom, 4 snapshot trimestrali 2024)
- **`load_ny_zip`** — legge gli snapshot NY da file zip Kaggle (data assegnata manualmente)
- **`load_insideairbnb_2025`** — legge i file `.csv.gz` Inside Airbnb 2025 (formato ricco con `last_scraped`)


In [4]:
def load_italy_2024(csv_path, cities):
    """
    Carica data/raw_2023_2024/airbnb.csv e filtra per le città richieste.
    Periodo: 4 snapshot trimestrali (2024-03, 2024-06, 2024-09, 2024-12).
    Nota: availability_365 e minimum_nights non presenti in questa fonte → NaN.
    """
    df = pd.read_csv(csv_path)
    df = df[df['City'].isin(cities)].copy()
    df = df.rename(columns={
        'Listings id':       'id',
        'City':              'city',
        'Neighbourhood':     'neighbourhood',
        'Property type':     'room_type',
        'Price':             'price',
        'Date of scraping':  'date',
        'Total reviews':     'number_of_reviews',
        'Last year reviews': 'number_of_reviews_ltm',
        'Reviews per month': 'reviews_per_month',
    })
    df['date'] = pd.to_datetime(df['date'])
    df['source'] = 'insideairbnb_2024'
    df['availability_365'] = pd.NA
    df['minimum_nights'] = pd.NA
    return df.reindex(columns=FINAL_COLS)


def load_ny_zip(zip_path, approx_date):
    """
    Carica uno snapshot New York da file zip (2023 o 2024).
    La data di scraping è assegnata manualmente — il file non contiene last_scraped,
    viene stimata in base al range di last_review presente nel dataset.
    """
    with zipfile.ZipFile(zip_path) as z:
        with z.open(z.namelist()[0]) as f:
            df = pd.read_csv(f, low_memory=False)
    df['city'] = 'New York'
    df['date'] = pd.to_datetime(approx_date)
    df['source'] = 'kaggle_zip'
    if 'number_of_reviews_ltm' not in df.columns:
        df['number_of_reviews_ltm'] = pd.NA
    return df.reindex(columns=FINAL_COLS)


def load_insideairbnb_2025(file_list, city_name):
    """
    Carica snapshot mensili da file .csv.gz Inside Airbnb 2025.
    Formato ricco: last_scraped, availability_365, minimum_nights, reviews, ecc.
    """
    dfs = []
    for f in file_list:
        df = pd.read_csv(f, usecols=[
            'id', 'neighbourhood_cleansed', 'room_type', 'price',
            'last_scraped', 'number_of_reviews', 'number_of_reviews_ltm',
            'reviews_per_month', 'availability_365', 'minimum_nights'
        ], low_memory=False)
        if df['price'].dtype == object:
            df['price'] = df['price'].str.replace(r'[\$,]', '', regex=True).astype(float)
        df = df.rename(columns={
            'neighbourhood_cleansed': 'neighbourhood',
            'last_scraped':           'date',
        })
        df['date'] = pd.to_datetime(df['date'])
        df['city'] = city_name
        df['source'] = 'insideairbnb_2025'
        dfs.append(df.reindex(columns=FINAL_COLS))
    return pd.concat(dfs, ignore_index=True)


print("Funzioni caricate.")


Funzioni caricate.


## 1. Dati italiani 2024 — `airbnb.csv`

Fonte: `data/raw_2023_2024/airbnb.csv`  
Periodo: 4 snapshot trimestrali (mar · giu · set · dic 2024)  
⚠️ `availability_365` e `minimum_nights` **non disponibili** in questa fonte → `NaN`


In [5]:
print("Caricamento airbnb.csv (Italia 2024)...")
df_italy = load_italy_2024('data/raw_2023_2024/airbnb.csv', CITIES_IT)

df_roma_24    = df_italy[df_italy['city'] == 'Roma'].copy()
df_milano_24  = df_italy[df_italy['city'] == 'Milano'].copy()
df_firenze_24 = df_italy[df_italy['city'] == 'Firenze'].copy()

for label, df in [('Roma', df_roma_24), ('Milano', df_milano_24), ('Firenze', df_firenze_24)]:
    dates = sorted(df['date'].dt.strftime('%Y-%m').unique())
    print(f"  {label:<10}: {len(df):>6} righe | {len(dates)} snapshot → {dates}")


Caricamento airbnb.csv (Italia 2024)...
  Roma      : 104981 righe | 4 snapshot → ['2024-03', '2024-06', '2024-09', '2024-12']
  Milano    :  75889 righe | 4 snapshot → ['2024-03', '2024-06', '2024-09', '2024-12']
  Firenze   :  41243 righe | 4 snapshot → ['2024-03', '2024-06', '2024-09', '2024-12']


## 2. Dati 2025 — Inside Airbnb (`.csv.gz`)

Fonte: `data/raw_2025/<city>/*.csv.gz`  
Formato ricco con `last_scraped`, `availability_365`, `minimum_nights`, ecc.

| Città | Snapshot disponibili |
|-------|---------------------|
| Roma | mar · giu · lug · set 2025 (4 snap.) |
| Milano | giu · set 2025 (2 snap.) |
| Firenze | mar · giu · set 2025 (3 snap.) |
| New York | mar–set + nov–dic 2025 (9 snap.) |


In [11]:
print("Caricamento dati 2025 (Inside Airbnb)...")

df_roma_25    = load_insideairbnb_2025(sorted(glob.glob('data/raw_2025/Roma/*.csv.gz')),     'Roma')
df_milano_25  = load_insideairbnb_2025(sorted(glob.glob('data/raw_2025/Milano/*.csv.gz')),  'Milano')
df_firenze_25 = load_insideairbnb_2025(sorted(glob.glob('data/raw_2025/Firenze/*.csv.gz')), 'Firenze')
df_ny_25      = load_insideairbnb_2025(sorted(glob.glob('data/raw_2025/NewYork/*.csv.gz')), 'New York')

for label, df in [('Roma', df_roma_25), ('Milano', df_milano_25), ('Firenze', df_firenze_25), ('New York', df_ny_25)]:
    dates = sorted(df['date'].dt.strftime('%Y-%m').unique())
    print(f"  {label:<12}: {len(df):>6} righe | {len(dates)} snapshot → {dates}")


Caricamento dati 2025 (Inside Airbnb)...
  Roma        : 108503 righe | 4 snapshot → ['2025-03', '2025-06', '2025-07', '2025-09']
  Milano      :  45473 righe | 2 snapshot → ['2025-06', '2025-09']
  Firenze     :  38786 righe | 3 snapshot → ['2025-03', '2025-06', '2025-09']
  New York    : 377709 righe | 9 snapshot → ['2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-11', '2025-12']


## 3. Dati New York 2023-2024 — file zip Kaggle

Fonte: `data/raw_2023_2024/NYairbnb2023.zip` e `NYairbnb2024.zip`  
⚠️ Nessun campo `last_scraped` — la data è assegnata manualmente:  
- `NYairbnb2023.zip` → data assegnata: **2023-03** (stimata da `last_review`)  
- `NYairbnb2024.zip` → data assegnata: **2024-01** (stimata da `last_review`)


In [7]:
print("Caricamento New York 2023-2024 (zip)...")

df_ny_23 = load_ny_zip('data/raw_2023_2024/NYairbnb2023.zip', '2023-03-01')
df_ny_24 = load_ny_zip('data/raw_2023_2024/NYairbnb2024.zip', '2024-01-01')

print(f"  NY 2023: {len(df_ny_23):>6} annunci | snapshot assegnato: 2023-03  ⚠️ approssimato")
print(f"  NY 2024: {len(df_ny_24):>6} annunci | snapshot assegnato: 2024-01  ⚠️ approssimato")


Caricamento New York 2023-2024 (zip)...
  NY 2023:  42931 annunci | snapshot assegnato: 2023-03  ⚠️ approssimato
  NY 2024:  20758 annunci | snapshot assegnato: 2024-01  ⚠️ approssimato


## 4. Salvataggio CSV per città

Unione di tutti gli snapshot disponibili per ciascuna città → `data/processed/<city>.csv`  
I file contengono **tutte le tipologie** di alloggio (`room_type`); il filtro sarà applicato solo nel dataset master.


In [12]:
city_dfs = {
    'Roma':     pd.concat([df_roma_24,    df_roma_25],                   ignore_index=True),
    'Milano':   pd.concat([df_milano_24,  df_milano_25],                  ignore_index=True),
    'Firenze':  pd.concat([df_firenze_24, df_firenze_25],                 ignore_index=True),
    'New York': pd.concat([df_ny_23,      df_ny_24,     df_ny_25],       ignore_index=True),
}

print("Salvataggio CSV per città:\n")
for city, df in city_dfs.items():
    df = df.sort_values('date').reset_index(drop=True)
    city_dfs[city] = df  # aggiorna con df ordinato
    fname = city.lower().replace(' ', '_')
    path = f'{OUTPUT_DIR}/{fname}.csv'
    df.to_csv(path, index=False)

    dates = sorted(df['date'].dt.strftime('%Y-%m').unique())
    room_counts = dict(df['room_type'].value_counts())
    print(f"  ✓ {city:<12} → {path}")
    print(f"    {len(df):>6} righe | {len(dates)} snapshot | {dates[0]} → {dates[-1]}")
    print(f"    room_type: {room_counts}")
    print()


Salvataggio CSV per città:

  ✓ Roma         → data/processed/roma.csv
    213484 righe | 8 snapshot | 2024-03 → 2025-09
    room_type: {'Entire home/apt': np.int64(81550), 'Entire home': np.int64(78198), 'Private room': np.int64(50291), 'Hotel room': np.int64(2774), 'Shared room': np.int64(671)}

  ✓ Milano       → data/processed/milano.csv
    121362 righe | 6 snapshot | 2024-03 → 2025-09
    room_type: {'Entire home': np.int64(65468), 'Entire home/apt': np.int64(39885), 'Private room': np.int64(15218), 'Shared room': np.int64(695), 'Hotel room': np.int64(96)}

  ✓ Firenze      → data/processed/firenze.csv
     80029 righe | 7 snapshot | 2024-03 → 2025-09
    room_type: {'Entire home': np.int64(34398), 'Entire home/apt': np.int64(32615), 'Private room': np.int64(12396), 'Hotel room': np.int64(486), 'Shared room': np.int64(134)}

  ✓ New York     → data/processed/new_york.csv
    441398 righe | 11 snapshot | 2023-03 → 2025-12
    room_type: {'Entire home/apt': np.int64(241867), 'Priva

## 5. Dataset master per il modello ARIMAX

Unione di tutte le città, filtrato su **`Entire home/apt`** (interi appartamenti).  
Questo è il dataset di input per la modellazione della serie temporale.

Salvato in: `data/processed/master_airbnb.csv`


In [13]:
df_master = pd.concat(city_dfs.values(), ignore_index=True)

# Normalizziamo: "Entire home" (fonte italia 2024) == "Entire home/apt" (Inside Airbnb)
df_master['room_type'] = df_master['room_type'].replace('Entire home', 'Entire home/apt')
df_master = df_master[df_master['room_type'] == 'Entire home/apt'].copy()

# Pulizia price: coerce eventuali stringhe residue (es. NY zip)
df_master['price'] = pd.to_numeric(
    df_master['price'].astype(str).str.replace(r'[\$,]', '', regex=True),
    errors='coerce'
)

df_master = df_master.sort_values(['city', 'date']).reset_index(drop=True)

master_path = f'{OUTPUT_DIR}/master_airbnb.csv'
df_master.to_csv(master_path, index=False)

print(f"Master dataset salvato: {master_path}")
print(f"Righe totali (Entire home/apt): {len(df_master)}\n")

# Snapshot per città
print("Snapshot per città:")
for city in sorted(df_master['city'].unique()):
    sub = df_master[df_master['city'] == city]
    dates = sorted(sub['date'].dt.strftime('%Y-%m').unique())
    print(f"  {city:<12}: {len(sub):>6} righe | {len(dates):>2} snapshot → {dates[0]} ... {dates[-1]}")

# Statistiche prezzo
print("\nPrezzo medio per città (Entire home/apt):")
print(df_master.groupby('city')['price'].agg(['mean', 'median', 'std']).round(2).to_string())

print("\nColonne:", list(df_master.columns))
print(f"\nDone! File pronti in: {os.path.abspath(OUTPUT_DIR)}")


Master dataset salvato: data/processed/master_airbnb.csv
Righe totali (Entire home/apt): 573981

Snapshot per città:
  Firenze     :  67013 righe |  7 snapshot → 2024-03 ... 2025-09
  Milano      : 105353 righe |  6 snapshot → 2024-03 ... 2025-09
  New York    : 241867 righe | 11 snapshot → 2023-03 ... 2025-12
  Roma        : 159748 righe |  8 snapshot → 2024-03 ... 2025-09

Prezzo medio per città (Entire home/apt):
            mean  median     std
city                            
Firenze   243.25   142.0  996.28
Milano    186.98   114.0  916.23
New York  283.11   192.0  645.17
Roma      197.47   136.0  487.37

Colonne: ['id', 'city', 'neighbourhood', 'room_type', 'price', 'date', 'number_of_reviews', 'number_of_reviews_ltm', 'reviews_per_month', 'availability_365', 'minimum_nights', 'source']

Done! File pronti in: c:\Users\user\airbnb\AirBnB-effect-study-with-ARIMAX-model\data\processed
